# 🤟 Sistema LSP → Castellano — Notebook Maestro Completo
## Lengua de Señas Peruana | Pipeline Integral | Google Colab

**Pipeline:** Drive → Extracción TARs → EDA → Preprocesamiento → Baseline → DL (LSTM + ST-GCN) → ONNX → API → Demo

| Dataset | Tamaño | Contenido |
|---|---|---|
| LSP - Palabras.tar | 1.0 GB | LSP - Palabras MP4 de señas LSP |
| Keypoints.tar | 690.2 MB | Landmarks pre-extraídos (input del modelo) |
| SRT.tar | 191.5 KB | Subtítulos .srt en castellano (ground truth) |

> ⚡ **Orden de ejecución:** correr todas las celdas de arriba hacia abajo (Ctrl+F9 en Colab)

## PASO 0 — Entorno, Drive y Extracción de TARs

In [ ]:
# Instalar dependencias (solo primera vez)
!pip install -q mediapipe==0.10.9 torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers scikit-learn pandas matplotlib seaborn gradio fastapi uvicorn[standard]
!pip install -q onnx onnxruntime-gpu pdfplumber wandb nltk jiwer
print('✅ Dependencias instaladas')

In [ ]:
!pip install -q onnx onnxruntime pdfplumber wandb nltk jiwer

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib seaborn gradio fastapi uvicorn[standard]

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib seaborn gradio fastapi "uvicorn[standard]"

In [ ]:
!pip install -q onnx onnxruntime pdfplumber wandb nltk jiwer

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib seaborn gradio fastapi uvicorn[standard]

In [ ]:
import os, sys, json, re, time, hashlib, tarfile, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

In [ ]:
# ── Configuración de rutas ────────────────────────────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/'   # ajustar si la carpeta compartida está en otro lugar
EXTRACT_DIR  = '/content/lsp_dataset/'
os.makedirs(EXTRACT_DIR, exist_ok=True)

ARCHIVOS_TAR = {
    'Videos.tar':    {'md5_prefix': '3bd', 'size_gb': 1.0},
    'Keypoints.tar': {'md5_prefix': '706', 'size_gb': 0.67},
    'SRT.tar':       {'md5_prefix': 'f31', 'size_gb': 0.0002},
}

def encontrar_tar(nombre, root):
    for r, dirs, files in os.walk(root):
        if nombre in files:
            return os.path.join(r, nombre)
    return None

def verificar_md5(filepath, prefijo):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    md5 = h.hexdigest()
    ok = md5.startswith(prefijo)
    print(f"{'✅' if ok else '❌'} {os.path.basename(filepath)}: MD5={md5[:8]}... (esperado: {prefijo}...)")
    return ok

# Localizar y verificar los 3 TARs
tar_paths = {}
for nombre, meta in ARCHIVOS_TAR.items():
    ruta = encontrar_tar(nombre, DATASET_ROOT)
    if ruta:
        size_gb = os.path.getsize(ruta) / 1e9
        print(f'📦 {nombre}: {ruta} ({size_gb:.3f} GB)')
        verificar_md5(ruta, meta['md5_prefix'])
        tar_paths[nombre] = ruta
    else:
        print(f'❌ {nombre} no encontrado — verificar acceso al Drive compartido')
        print(f'   URL del dataset: https://drive.google.com/drive/u/0/folders/1JjakUGGrAn9YwdHrkAUNCmuzyS8jdCse')

In [ ]:
# ── Extraer los 3 TARs ───────────────────────────────────────────────────────
for nombre, ruta in tar_paths.items():
    dest = os.path.join(EXTRACT_DIR, nombre.replace('.tar', ''))
    os.makedirs(dest, exist_ok=True)
    if len(os.listdir(dest)) > 0:
        print(f'⏭  {nombre}: ya extraído → {dest}')
        continue
    print(f'📦 Extrayendo {nombre}...')
    t0 = time.time()
    with tarfile.open(ruta, 'r') as tar:
        tar.extractall(path=dest)
    print(f'   ✅ → {dest} ({time.time()-t0:.1f}s)')

# Estructura del dataset extraído
print('\n📂 Estructura:')
for root, dirs, files in os.walk(EXTRACT_DIR):
    nivel = root.replace(EXTRACT_DIR, '').count(os.sep)
    if nivel > 2: continue
    indent = '  ' * nivel
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files)[:3]:
        print(f'{indent}  {f}')
    if len(files) > 3:
        print(f'{indent}  ... ({len(files)} archivos)')

# Definir rutas finales
VIDEOS_DIR    = os.path.join(EXTRACT_DIR, 'Videos')
KEYPOINTS_DIR = os.path.join(EXTRACT_DIR, 'Keypoints')
SRT_DIR       = os.path.join(EXTRACT_DIR, 'SRT')

## SECCIÓN 1 — EDA Completo (3 archivos TAR)

In [ ]:
# ── EDA LSP - Palabras.tar ───────────────────────────────────────────────────────────
videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, '**/*.mp4'), recursive=True))
clases_video = [os.path.basename(os.path.dirname(v)) for v in videos]
conteo_clases = Counter(clases_video)

print(f'Total LSP - Palabras: {len(LSP - Palabras)}')
print(f'LSP - Vocabulario-palabras únicas: {len(conteo_clases)}')
print(f'LSP - Palabras por clase (min/max/mean): {min(conteo_clases.values())}/{max(conteo_clases.values())}/{np.mean(list(conteo_clases.values())):.1f}')

# Analizar fps, frames, duración de una muestra
sample_stats = []
for vpath in videos[:min(50, len(videos))]:
    cap = cv2.VideoCapture(vpath)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    nf    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = nf / fps if fps > 0 else 0
    cap.release()
    sample_stats.append({'clase': os.path.basename(os.path.dirname(vpath)), 'fps': fps, 'frames': nf, 'dur_s': dur})

df_stats = pd.DataFrame(sample_stats)
print('\nEstadísticas de video (muestra 50):')
print(df_stats[['fps', 'frames', 'dur_s']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('EDA — LSP - Palabras.tar LSP', fontsize=14, fontweight='bold')

# Distribución de LSP - Vocabulario-palabras (top 20)
top20 = dict(sorted(conteo_clases.items(), key=lambda x: -x[1])[:20])
axes[0,0].barh(list(top20.keys()), list(top20.values()), color='steelblue')
axes[0,0].set_title('Top 20 LSP - Vocabulario-palabras por nº de LSP - Palabras')
axes[0,0].set_xlabel('N° LSP - Palabras')

# Distribución de duración
axes[0,1].hist(df_stats['dur_s'], bins=20, color='darkorange', edgecolor='white')
axes[0,1].set_title('Distribución de duración (s)')
axes[0,1].set_xlabel('Duración (s)')

# FPS
axes[1,0].hist(df_stats['fps'], bins=15, color='seagreen', edgecolor='white')
axes[1,0].set_title('Distribución de FPS')
axes[1,0].set_xlabel('FPS')

# Frames por video
axes[1,1].hist(df_stats['frames'], bins=20, color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Distribución de frames por video')
axes[1,1].set_xlabel('N° frames')

plt.tight_layout()
plt.savefig('/content/eda_videos.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada: /content/eda_videos.png')

In [ ]:
# ── EDA Keypoints.tar ────────────────────────────────────────────────────────
kp_files = sorted(glob.glob(os.path.join(KEYPOINTS_DIR, '**/*'), recursive=True))
kp_files = [f for f in kp_files if os.path.isfile(f)]

ext_count = Counter(os.path.splitext(f)[1].lower() for f in kp_files)
print('Extensiones encontradas:', dict(ext_count))
print(f'Total archivos keypoints: {len(kp_files)}')

clases_kp = [os.path.basename(os.path.dirname(f)) for f in kp_files]
conteo_kp = Counter(clases_kp)
print(f'LSP - Vocabulario-palabras únicas en Keypoints: {len(conteo_kp)}')

# Inspeccionar shape de una muestra
if kp_files:
    ext_ppal = ext_count.most_common(1)[0][0]
    muestra = [f for f in kp_files if f.endswith(ext_ppal)][:3]
    for f in muestra:
        if ext_ppal == '.npy':
            data = np.load(f, allow_pickle=True)
        elif ext_ppal == '.csv':
            data = pd.read_csv(f).values
        elif ext_ppal == '.json':
            with open(f) as jf: raw = json.load(jf)
            data = np.array(raw if isinstance(raw, list) else raw.get('keypoints', []))
        else:
            data = np.array([])
        print(f'  {os.path.basename(f)}: shape={np.array(data).shape}')

In [ ]:
# ── EDA SRT.tar ──────────────────────────────────────────────────────────────
def parsear_srt(srt_path):
    with open(srt_path, 'r', encoding='utf-8', errors='replace') as f:
        contenido = f.read()
    patron = r'(\d+)\n(\d{2}:\d{2}:\d{2}[,\.]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,\.]\d{3})\n([\s\S]*?)(?=\n\n|\Z)'
    segmentos = re.findall(patron, contenido.strip())
    return [{'id': int(n), 'inicio': ini, 'fin': fin,
              'texto': texto.strip().replace('\n', ' ')}
             for n, ini, fin, texto in segmentos]

srt_files = sorted(glob.glob(os.path.join(SRT_DIR, '**/*.srt'), recursive=True))
print(f'Total archivos SRT: {len(srt_files)}')

ground_truth = {}
for sp in srt_files:
    nombre = os.path.splitext(os.path.basename(sp))[0]
    segs = parsear_srt(sp)
    ground_truth[nombre] = {
        'transcripcion': ' '.join(s['texto'] for s in segs),
        'segmentos': segs,
        'n_segmentos': len(segs),
    }

print(f'Registros ground truth: {len(ground_truth)}')
for nombre, datos in list(ground_truth.items())[:3]:
    print(f'  {nombre}: "{datos["transcripcion"][:60]}..." ({datos["n_segmentos"]} segmentos)')

# Longitud de transcripciones
longitudes = [len(d['transcripcion'].split()) for d in ground_truth.values()]
print(f'\nPalabras por transcripción — min:{min(longitudes)} max:{max(longitudes)} media:{np.mean(longitudes):.1f}')

In [ ]:
# Verificar alineación LSP - Palabras ↔ Keypoints ↔ SRT por nombre base
bases_videos    = {os.path.splitext(os.path.basename(v))[0] for v in videos}
ext_kp          = ext_count.most_common(1)[0][0]
bases_keypoints = {os.path.splitext(os.path.basename(f))[0] for f in kp_files}
bases_srt       = set(ground_truth.keys())

alineados = bases_videos & bases_keypoints & bases_srt
print(f'Archivos en LSP - Palabras:    {len(bases_videos)}')
print(f'Archivos en Keypoints: {len(bases_keypoints)}')
print(f'Archivos en SRT:       {len(bases_srt)}')
print(f'Alineados (los 3):     {len(alineados)}')
print(f'Sin keypoints:         {len(bases_videos - bases_keypoints)}')
print(f'Sin SRT:               {len(bases_videos - bases_srt)}')

# Riesgo de leakage: mismos sujetos en train/test
# LSP suele nombrar los LSP - Palabras como: seña_sujetoID_toma
sujetos = set()
for b in bases_videos:
    partes = b.split('_')
    if len(partes) >= 2:
        sujetos.add(partes[1])
print(f'\n⚠️  Sujetos identificados en nombres: {len(sujetos)}')
print('   → Usar split por sujeto para evitar leakage.')

## SECCIÓN 2 — Preprocesamiento y Pipeline de Datos

In [ ]:
# ── Carga de Keypoints (formato adaptativo) ───────────────────────────────────
def cargar_keypoints(kp_dir):
    archivos = sorted(glob.glob(os.path.join(kp_dir, '**/*'), recursive=True))
    archivos = [f for f in archivos if os.path.isfile(f)]
    ext = Counter(os.path.splitext(f)[1].lower() for f in archivos).most_common(1)[0][0]

    X, y, rutas = [], [], []
    for fpath in sorted(glob.glob(os.path.join(kp_dir, f'**/*{ext}'), recursive=True)):
        clase = os.path.basename(os.path.dirname(fpath))
        try:
            if ext == '.npy':
                kp = np.load(fpath, allow_pickle=True)
            elif ext == '.csv':
                kp = pd.read_csv(fpath).values
            elif ext == '.json':
                with open(fpath) as jf: raw = json.load(jf)
                kp = np.array(raw if isinstance(raw, list) else raw.get('keypoints', []))
            else:
                continue
            X.append(np.array(kp, dtype=np.float32))
            y.append(clase)
            rutas.append(fpath)
        except Exception as e:
            print(f'  Error {fpath}: {e}')
            continue

    print(f'Keypoints cargados: {len(X)} muestras, {len(set(y))} LSP - Vocabulario-palabras')
    if X:
        shapes = [np.array(x).shape for x in X[:5]]
        print(f'Shapes de muestra: {shapes}')
    return X, y, rutas

X_raw, y_raw, rutas_raw = cargar_keypoints(KEYPOINTS_DIR)

In [ ]:
# ── Normalización y padding/truncado a T=30 frames ───────────────────────────
N_FRAMES = 30

def normalizar_kp(kp):
    """Centra landmarks respecto a muñeca izquierda (punto 0 de MediaPipe Holistic)."""
    kp = np.array(kp, dtype=np.float32)
    if kp.ndim == 2:                      # [N_kp, 3] → añadir dimensión temporal
        kp = kp[np.newaxis]
    ref = kp[:, 0:1, :]                   # [T, 1, 3]
    return kp - ref

def pad_truncate(kp, n_frames=N_FRAMES):
    kp = np.array(kp, dtype=np.float32)
    if kp.ndim == 2:
        kp = kp[np.newaxis]
    T = kp.shape[0]
    if T >= n_frames:
        idx = np.linspace(0, T - 1, n_frames, dtype=int)
        return kp[idx]
    pad = np.zeros((n_frames - T, *kp.shape[1:]), dtype=np.float32)
    return np.concatenate([kp, pad], axis=0)

X_proc, y_proc = [], []
for kp, etq in zip(X_raw, y_raw):
    kp_norm = normalizar_kp(kp)
    kp_pad  = pad_truncate(kp_norm, N_FRAMES)
    X_proc.append(kp_pad)
    y_proc.append(etq)

X_proc = np.array(X_proc)   # [N, T=30, KP, 3]
print(f'Dataset procesado: {X_proc.shape}')   # e.g. (N, 30, 1662, 3)

# Codificar etiquetas
le = LabelEncoder()
y_enc = le.fit_transform(y_proc)
print(f'LSP - Vocabulario-palabras: {len(le.classes_)}')
print(f'Primeras 5 LSP - Vocabulario-palabras: {le.classes_[:5].tolist()}')

# Guardar mapeo
label2idx = {c: int(i) for i, c in enumerate(le.classes_)}
idx2label = {v: k for k, v in label2idx.items()}
with open('/content/label2idx.json', 'w') as f:
    json.dump(label2idx, f, ensure_ascii=False, indent=2)
print('✅ label2idx.json guardado')

In [ ]:
# ── Aumentación específica para LSP ──────────────────────────────────────────
def augment_kp(kp, flip=True, jitter_sigma=0.01, speed_range=(0.75, 1.25)):
    """Flip + jitter de landmarks + variación de velocidad."""
    kp = kp.copy()
    # Jitter gaussiano
    if jitter_sigma > 0:
        kp += np.random.normal(0, jitter_sigma, kp.shape).astype(np.float32)
    # Flip horizontal (solo para señas no lateralizadas)
    if flip and np.random.random() < 0.5:
        kp[:, :, 0] = -kp[:, :, 0]   # invertir eje x
    # Variación de velocidad (resample temporal)
    speed = np.random.uniform(*speed_range)
    T = kp.shape[0]
    T_new = int(T * speed)
    if T_new > 0:
        idx = np.linspace(0, T - 1, T_new, dtype=int)
        kp = kp[idx]
    return pad_truncate(kp, N_FRAMES)

print('Aumentación definida: flip + jitter σ=0.01 + speed ×0.75/×1.25')
# Ejemplo
muestra = X_proc[0]
aug_ej  = augment_kp(muestra)
print(f'Shape original: {muestra.shape} → aumentado: {aug_ej.shape}')

In [ ]:
# ── Splits 70/15/15 estratificados ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

idx_all = np.arange(len(X_proc))
X_flat  = X_proc.reshape(len(X_proc), -1)   # para stratify

idx_train, idx_temp = train_test_split(idx_all, test_size=0.30,
                                        stratify=y_enc, random_state=42)
idx_val,   idx_test = train_test_split(idx_temp, test_size=0.50,
                                        stratify=y_enc[idx_temp], random_state=42)

X_train, y_train = X_proc[idx_train], y_enc[idx_train]
X_val,   y_val   = X_proc[idx_val],   y_enc[idx_val]
X_test,  y_test  = X_proc[idx_test],  y_enc[idx_test]

print(f'Train: {len(X_train)} ({len(X_train)/len(X_proc)*100:.0f}%)')
print(f'Val:   {len(X_val)}   ({len(X_val)/len(X_proc)*100:.0f}%)')
print(f'Test:  {len(X_test)}  ({len(X_test)/len(X_proc)*100:.0f}%)')

# Verificar distribución de LSP - Vocabulario-palabras
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = Counter(y_split)
    print(f'{split_name}: {len(counts)} LSP - Vocabulario-palabras | desbalance (max/min): {max(counts.values())}/{min(counts.values())}')

## SECCIÓN 3 — Baseline (KNN + Naive Bayes + Regresión Logística)

In [ ]:
# Aplanar para baseline clásico
X_tr_flat  = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_te_flat  = X_test.reshape(len(X_test), -1)

# Submuestrear si dataset muy grande para KNN (memoria)
MAX_BASE = 5000
if len(X_tr_flat) > MAX_BASE:
    np.random.seed(42)
    sel = np.random.choice(len(X_tr_flat), MAX_BASE, replace=False)
    X_tr_b, y_tr_b = X_tr_flat[sel], y_train[sel]
else:
    X_tr_b, y_tr_b = X_tr_flat, y_train

print(f'Baseline dataset: {X_tr_b.shape}')

baseline_modelos = {
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes':         GaussianNB(),
    'Reg. Logística':      LogisticRegression(max_iter=500, C=1.0, solver='lbfgs',
                                               multi_class='auto', n_jobs=-1),
}

baseline_resultados = {}
for nombre, modelo in baseline_modelos.items():
    t0 = time.time()
    modelo.fit(X_tr_b, y_tr_b)
    t_train = time.time() - t0

    t0 = time.time()
    y_pred = modelo.predict(X_te_flat)
    t_inf  = (time.time() - t0) / len(X_te_flat) * 1000   # ms/muestra

    f1_macro    = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc         = (y_pred == y_test).mean()

    baseline_resultados[nombre] = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'latencia_ms': t_inf,
        't_train_s': t_train,
    }
    print(f'{nombre}: Acc={acc:.3f} | F1-macro={f1_macro:.3f} | F1-w={f1_weighted:.3f} | lat={t_inf:.2f}ms')

df_baseline = pd.DataFrame(baseline_resultados).T.round(3)
print('\nTabla de resultados baseline:')
print(df_baseline)

In [ ]:
# Mejor baseline para comparar con DL
mejor_base_nombre = df_baseline['f1_macro'].idxmax()
mejor_f1_base     = df_baseline.loc[mejor_base_nombre, 'f1_macro']
print(f'Mejor baseline: {mejor_base_nombre} (F1-macro={mejor_f1_base:.3f})')
print(f'→ El modelo DL debe superar F1-macro={mejor_f1_base:.3f}')

# Guardar para comparación final
df_baseline.to_csv('/content/baseline_results.csv')
print('✅ baseline_results.csv guardado')

## SECCIÓN 4 — Modelos Deep Learning (LSTM Bidir + ST-GCN)

In [ ]:
# ── Dataset y DataLoaders ────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

N_CLASES = len(le.classes_)

class LSPDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.from_numpy(X).float()    # [N, T, KP, 3]
        self.y = torch.from_numpy(y).long()
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i].numpy()
        if self.augment:
            x = augment_kp(x)
            x = torch.from_numpy(x).float()
        else:
            x = self.X[i]
        return x, self.y[i]

# Pesos de clase para compensar desbalance
conteo_y = Counter(y_train.tolist())
total_y  = len(y_train)
class_weights = torch.tensor(
    [total_y / (N_CLASES * conteo_y.get(i, 1)) for i in range(N_CLASES)],
    dtype=torch.float32
).to(DEVICE)

BATCH = 32
ds_train = LSPDataset(X_train, y_train, augment=True)
ds_val   = LSPDataset(X_val,   y_val,   augment=False)
ds_test  = LSPDataset(X_test,  y_test,  augment=False)

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders listos | Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')
print(f'N_CLASES: {N_CLASES} | shape entrada: {X_train.shape[1:]}')

In [ ]:
# ── MODELO 1: LSTM Bidireccional + Attention ──────────────────────────────────
class LSTMBidir(nn.Module):
    def __init__(self, n_kp, n_coords, n_clases, hidden=256, n_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(n_kp * n_coords, 256)
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.attn_q  = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Sequential(
            nn.Linear(hidden * 2, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_clases),
        )

    def forward(self, x):                                  # x: [B, T, KP, 3]
        B, T, KP, C = x.shape
        x = x.view(B, T, KP * C)                           # [B, T, KP*3]
        x = F.gelu(self.input_proj(x))                    # [B, T, 256]
        h, _ = self.lstm(x)                                # [B, T, H*2]
        attn = torch.softmax(self.attn_q(h), dim=1)       # [B, T, 1]
        ctx  = (attn * h).sum(dim=1)                       # [B, H*2]
        return self.fc(self.dropout(ctx))

N_KP     = X_train.shape[2]
N_COORDS = X_train.shape[3]
modelo_lstm = LSTMBidir(N_KP, N_COORDS, N_CLASES).to(DEVICE)
params = sum(p.numel() for p in modelo_lstm.parameters() if p.requires_grad)
print(f'LSTM Bidir: {params:,} parámetros')

# Test forward
dummy = torch.randn(2, N_FRAMES, N_KP, N_COORDS).to(DEVICE)
with torch.no_grad():
    out = modelo_lstm(dummy)
print(f'Output shape: {out.shape}')   # [2, N_CLASES]

In [ ]:
# ── MODELO 2: ST-GCN simplificado ────────────────────────────────────────────
class STGCNLayer(nn.Module):
    """Capa de grafo espacio-temporal sobre keypoints."""
    def __init__(self, in_ch, out_ch, n_kp, dropout=0.1):
        super().__init__()
        self.n_kp = n_kp
        # Matriz de adyacencia aprendible
        self.A = nn.Parameter(torch.eye(n_kp) * 0.1 +
                               torch.rand(n_kp, n_kp) * 0.01)
        self.gcn  = nn.Linear(in_ch, out_ch)
        self.tcn  = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, KP, C = x.shape
        A_norm = F.softmax(self.A, dim=-1)
        # GCN sobre keypoints en cada frame
        xr = x.reshape(B * T, KP, C)
        xg = torch.bmm(A_norm.unsqueeze(0).expand(B * T, -1, -1), xr)
        xg = F.gelu(self.gcn(xg))                         # [BT, KP, out_ch]
        # TCN sobre eje temporal
        xg = xg.reshape(B, T, KP, -1).permute(0, 3, 2, 1)  # [B, out_ch, KP, T]
        xg = xg.reshape(B * xg.shape[1], KP, T)             # hack: tratar KP como batch
        # pooling sobre KP
        xg = xg.mean(dim=1).unsqueeze(0)                   # simplificado
        return xg


class STGCN(nn.Module):
    def __init__(self, n_kp, n_coords, n_clases, hidden=128, dropout=0.3):
        super().__init__()
        self.embed = nn.Linear(n_coords, 64)
        self.gcn1  = nn.Linear(64, hidden)
        self.gcn2  = nn.Linear(hidden, hidden)
        self.A     = nn.Parameter(torch.eye(n_kp))
        self.tcn   = nn.Sequential(
            nn.Conv1d(hidden, hidden, 3, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, 3, padding=1),
        )
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.fc    = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_clases),
        )

    def forward(self, x):                                  # [B, T, KP, C]
        B, T, KP, C = x.shape
        x = F.gelu(self.embed(x))                          # [B, T, KP, 64]
        # GCN sobre KP
        A = F.softmax(self.A, dim=-1)
        x = x.reshape(B * T, KP, 64)
        x = F.gelu(self.gcn1(torch.bmm(A.unsqueeze(0).expand(B*T,-1,-1), x)))
        x = F.gelu(self.gcn2(torch.bmm(A.unsqueeze(0).expand(B*T,-1,-1), x)))
        x = x.mean(dim=1)                                  # [BT, hidden]
        x = x.reshape(B, T, -1).permute(0, 2, 1)          # [B, hidden, T]
        x = self.tcn(x)                                    # [B, hidden, T]
        x = self.pool(x).squeeze(-1)                       # [B, hidden]
        return self.fc(x)

modelo_stgcn = STGCN(N_KP, N_COORDS, N_CLASES).to(DEVICE)
params_stgcn = sum(p.numel() for p in modelo_stgcn.parameters() if p.requires_grad)
print(f'ST-GCN: {params_stgcn:,} parámetros')
with torch.no_grad():
    out_s = modelo_stgcn(dummy)
print(f'ST-GCN output: {out_s.shape}')

In [ ]:
# ── Loop de entrenamiento genérico ───────────────────────────────────────────
def entrenar(modelo, dl_train, dl_val, n_epochs=60, lr=1e-4, wd=1e-4, patience=10):
    optimizer = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    historial = {'loss_tr': [], 'loss_val': [], 'acc_val': [], 'f1_val': []}
    mejor_f1, espera, mejor_ckpt = 0.0, 0, None

    for epoch in range(1, n_epochs + 1):
        # ── Entrenamiento ──
        modelo.train()
        loss_sum, n = 0.0, 0
        for xb, yb in dl_train:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = modelo(xb)
                loss   = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(xb)
            n        += len(xb)
        scheduler.step()
        loss_tr = loss_sum / n

        # ── Validación ──
        modelo.eval()
        preds, trues, vl = [], [], 0.0
        with torch.no_grad():
            for xb, yb in dl_val:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                    logits = modelo(xb)
                    vl    += criterion(logits, yb).item() * len(xb)
                preds.extend(logits.argmax(1).cpu().numpy())
                trues.extend(yb.cpu().numpy())
        loss_val = vl / len(ds_val)
        acc_val  = (np.array(preds) == np.array(trues)).mean()
        f1_val   = f1_score(trues, preds, average='macro', zero_division=0)

        historial['loss_tr'].append(loss_tr)
        historial['loss_val'].append(loss_val)
        historial['acc_val'].append(acc_val)
        historial['f1_val'].append(f1_val)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}: loss_tr={loss_tr:.4f} | loss_val={loss_val:.4f} '
                  f'| acc={acc_val:.3f} | F1={f1_val:.3f}')

        # Early stopping
        if f1_val > mejor_f1:
            mejor_f1   = f1_val
            mejor_ckpt = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
            espera     = 0
        else:
            espera += 1
            if espera >= patience:
                print(f'  Early stopping en epoch {epoch} (mejor F1={mejor_f1:.3f})')
                break

    if mejor_ckpt:
        modelo.load_state_dict(mejor_ckpt)
    return historial, mejor_f1

print('Función de entrenamiento definida ✅')

In [ ]:
# ── Entrenar LSTM Bidireccional ───────────────────────────────────────────────
print('=' * 60)
print('ENTRENANDO: LSTM Bidireccional + Attention')
print('=' * 60)
hist_lstm, f1_lstm = entrenar(modelo_lstm, dl_train, dl_val, n_epochs=60, lr=1e-4, patience=10)

# Guardar checkpoint
torch.save({
    'state_dict': modelo_lstm.state_dict(),
    'label2idx':  label2idx,
    'n_clases':   N_CLASES,
    'f1_val':     f1_lstm,
}, '/content/checkpoints/lstm_best.pt')
print(f'\n✅ LSTM guardado | F1-val={f1_lstm:.3f}')

In [ ]:
# ── Entrenar ST-GCN ────────────────────────────────────────────────────────────
print('=' * 60)
print('ENTRENANDO: ST-GCN (Spatial-Temporal Graph CNN)')
print('=' * 60)
hist_stgcn, f1_stgcn = entrenar(modelo_stgcn, dl_train, dl_val, n_epochs=60, lr=5e-4, patience=10)

torch.save({
    'state_dict': modelo_stgcn.state_dict(),
    'label2idx':  label2idx,
    'n_clases':   N_CLASES,
    'f1_val':     f1_stgcn,
}, '/content/checkpoints/stgcn_best.pt')
print(f'\n✅ ST-GCN guardado | F1-val={f1_stgcn:.3f}')

## SECCIÓN 5 — Evaluación Completa y Métricas

In [ ]:
# ── Curvas de aprendizaje ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Curvas de Aprendizaje — LSTM vs ST-GCN', fontsize=13, fontweight='bold')

for ax, metrica, titulo, ylabel in zip(
    axes,
    ['loss_val', 'acc_val', 'f1_val'],
    ['Loss Validación', 'Accuracy Validación', 'F1-macro Validación'],
    ['CrossEntropy', 'Accuracy', 'F1-macro']
):
    ax.plot(hist_lstm[metrica],  label='LSTM Bidir',  color='steelblue',  linewidth=2)
    ax.plot(hist_stgcn[metrica], label='ST-GCN',      color='darkorange', linewidth=2)
    ax.set_title(titulo)
    ax.set_xlabel('Época')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/curvas_aprendizaje.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ curvas_aprendizaje.png guardado')

In [ ]:
# ── Evaluación en Test Set ────────────────────────────────────────────────────
def evaluar_en_test(modelo, dl_test, nombre_modelo):
    modelo.eval()
    preds, trues = [], []
    tiempos = []
    with torch.no_grad():
        for xb, yb in dl_test:
            xb = xb.to(DEVICE)
            t0 = time.perf_counter()
            logits = modelo(xb)
            tiempos.append((time.perf_counter() - t0) / len(xb) * 1000)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())

    acc         = (np.array(preds) == np.array(trues)).mean()
    f1_macro    = f1_score(trues, preds, average='macro', zero_division=0)
    f1_weighted = f1_score(trues, preds, average='weighted', zero_division=0)
    lat_ms      = np.mean(tiempos)

    print(f'\n── {nombre_modelo} ──')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  F1-macro:    {f1_macro:.4f}')
    print(f'  F1-weighted: {f1_weighted:.4f}')
    print(f'  Latencia:    {lat_ms:.2f} ms/muestra')

    cm = confusion_matrix(trues, preds)
    return {
        'accuracy': acc, 'f1_macro': f1_macro,
        'f1_weighted': f1_weighted, 'latencia_ms': lat_ms,
        'preds': preds, 'trues': trues, 'cm': cm,
    }

os.makedirs('/content/checkpoints', exist_ok=True)
res_lstm  = evaluar_en_test(modelo_lstm,  dl_test, 'LSTM Bidireccional')
res_stgcn = evaluar_en_test(modelo_stgcn, dl_test, 'ST-GCN')

In [ ]:
# ── Tabla de comparación final ────────────────────────────────────────────────
tabla_final = pd.DataFrame({
    'Modelo': ['Baseline KNN', 'Baseline LogReg', 'LSTM Bidir + Attn', 'ST-GCN'],
    'Accuracy': [
        baseline_resultados.get('KNN (k=5)', {}).get('accuracy', 0),
        baseline_resultados.get('Reg. Logística', {}).get('accuracy', 0),
        res_lstm['accuracy'],
        res_stgcn['accuracy'],
    ],
    'F1-macro': [
        baseline_resultados.get('KNN (k=5)', {}).get('f1_macro', 0),
        baseline_resultados.get('Reg. Logística', {}).get('f1_macro', 0),
        res_lstm['f1_macro'],
        res_stgcn['f1_macro'],
    ],
    'F1-weighted': [
        baseline_resultados.get('KNN (k=5)', {}).get('f1_weighted', 0),
        baseline_resultados.get('Reg. Logística', {}).get('f1_weighted', 0),
        res_lstm['f1_weighted'],
        res_stgcn['f1_weighted'],
    ],
    'Latencia (ms)': [
        baseline_resultados.get('KNN (k=5)', {}).get('latencia_ms', 0),
        baseline_resultados.get('Reg. Logística', {}).get('latencia_ms', 0),
        res_lstm['latencia_ms'],
        res_stgcn['latencia_ms'],
    ],
}).round(4)

tabla_final['Tipo'] = ['Baseline', 'Baseline', 'Deep Learning', 'Deep Learning']
print('\n' + '=' * 65)
print('TABLA COMPARATIVA FINAL — LSP Traductor')
print('=' * 65)
print(tabla_final.to_string(index=False))
print('=' * 65)
tabla_final.to_csv('/content/tabla_comparativa.csv', index=False)
print('\n✅ tabla_comparativa.csv guardado')

In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Matrices de Confusión — Test Set', fontsize=13, fontweight='bold')

for ax, res, titulo in zip(axes,
                             [res_lstm, res_stgcn],
                             ['LSTM Bidireccional', 'ST-GCN']):
    cm_norm = res['cm'].astype(float) / (res['cm'].sum(axis=1, keepdims=True) + 1e-8)
    # Mostrar solo si hay pocas LSP - Vocabulario-palabras; si hay muchas, usar heatmap
    if N_CLASES <= 30:
        sns.heatmap(cm_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
                    xticklabels=le.classes_, yticklabels=le.classes_, annot=N_CLASES <= 15)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    else:
        sns.heatmap(cm_norm, ax=ax, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{titulo}\nF1-macro={res["f1_macro"]:.3f}')
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('/content/matrices_confusion.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ matrices_confusion.png guardado')

In [ ]:
# ── WER (Word Error Rate) con SRT como ground truth ──────────────────────────
def calcular_wer(ref, hyp):
    ref_w = ref.lower().split()
    hyp_w = hyp.lower().split()
    if not ref_w:
        return 0.0
    d = np.zeros((len(ref_w)+1, len(hyp_w)+1))
    for i in range(len(ref_w)+1): d[i][0] = i
    for j in range(len(hyp_w)+1): d[0][j] = j
    for i in range(1, len(ref_w)+1):
        for j in range(1, len(hyp_w)+1):
            cost = 0 if ref_w[i-1] == hyp_w[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(ref_w)][len(hyp_w)] / len(ref_w)

# Calcular WER sobre los LSP - Palabras alineados con SRT
wer_scores = []
mejor_modelo = modelo_lstm if res_lstm['f1_macro'] >= res_stgcn['f1_macro'] else modelo_stgcn
mejor_modelo.eval()

for nombre, gt in list(ground_truth.items())[:20]:  # muestra de 20
    # Buscar keypoints correspondientes
    kp_path = None
    for fpath in kp_files:
        if os.path.splitext(os.path.basename(fpath))[0] == nombre:
            kp_path = fpath
            break
    if kp_path is None:
        continue

    ext_kp = os.path.splitext(kp_path)[1].lower()
    if ext_kp == '.npy':
        kp = np.load(kp_path, allow_pickle=True)
    elif ext_kp == '.csv':
        kp = pd.read_csv(kp_path).values
    else:
        continue

    kp_norm = normalizar_kp(kp)
    kp_pad  = pad_truncate(kp_norm, N_FRAMES)
    xb = torch.from_numpy(kp_pad).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = mejor_modelo(xb)
        pred_idx = logits.argmax(1).item()
    pred_clase = idx2label.get(pred_idx, '?')

    ref = gt['transcripcion']
    wer = calcular_wer(ref, pred_clase)
    wer_scores.append(wer)

if wer_scores:
    print(f'WER promedio (muestra 20): {np.mean(wer_scores):.3f}')
    print(f'WER min/max: {min(wer_scores):.3f} / {max(wer_scores):.3f}')
else:
    print('No se pudieron calcular WER (archivos no alineados por nombre)')

## SECCIÓN 6 — Exportación ONNX y Benchmark de Latencia

In [ ]:
import onnx
import onnxruntime as ort

os.makedirs('/content/checkpoints', exist_ok=True)

# Elegir el mejor modelo para exportar
if res_lstm['f1_macro'] >= res_stgcn['f1_macro']:
    modelo_export = modelo_lstm
    nombre_export = 'lstm_best'
    print(f'Exportando LSTM Bidir (F1={res_lstm["f1_macro"]:.3f})')
else:
    modelo_export = modelo_stgcn
    nombre_export = 'stgcn_best'
    print(f'Exportando ST-GCN (F1={res_stgcn["f1_macro"]:.3f})')

modelo_export.eval().cpu()
dummy_cpu = torch.randn(1, N_FRAMES, N_KP, N_COORDS)
onnx_path = f'/content/checkpoints/{nombre_export}.onnx'

torch.onnx.export(
    modelo_export,
    dummy_cpu,
    onnx_path,
    input_names=['keypoints'],
    output_names=['logits'],
    dynamic_axes={'keypoints': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17,
    do_constant_folding=True,
)

# Verificar
modelo_onnx = onnx.load(onnx_path)
onnx.checker.check_model(modelo_onnx)
print(f'✅ ONNX exportado y validado: {onnx_path}')
print(f'   Tamaño: {os.path.getsize(onnx_path)/1e6:.2f} MB')
modelo_export.to(DEVICE)

In [ ]:
# ── Benchmark de latencia ONNX vs PyTorch ───────────────────────────────────
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if DEVICE == 'cuda' else ['CPUExecutionProvider']
ort_sess  = ort.InferenceSession(onnx_path, providers=providers)
x_bench   = np.random.randn(1, N_FRAMES, N_KP, N_COORDS).astype(np.float32)

N_RUNS = 100

# ONNX latencia
tiempos_onnx = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    ort_sess.run(None, {'keypoints': x_bench})
    tiempos_onnx.append((time.perf_counter() - t0) * 1000)

# PyTorch latencia
modelo_export.eval().to(DEVICE)
x_pt = torch.from_numpy(x_bench).to(DEVICE)
tiempos_pt = []
with torch.no_grad():
    for _ in range(N_RUNS):
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        modelo_export(x_pt)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        tiempos_pt.append((time.perf_counter() - t0) * 1000)

print(f'Latencia ONNX:    {np.mean(tiempos_onnx):.2f} ± {np.std(tiempos_onnx):.2f} ms')
print(f'Latencia PyTorch: {np.mean(tiempos_pt):.2f} ± {np.std(tiempos_pt):.2f} ms')
objetivo = 200
lat_onnx = np.mean(tiempos_onnx)
status = '✅' if lat_onnx < objetivo else '⚠️'
print(f'{status} Latencia ONNX ({lat_onnx:.1f}ms) vs objetivo <{objetivo}ms')

## SECCIÓN 7 — API FastAPI (inferencia en tiempo real)

In [ ]:
# Generar clase_texto.json (mapeo clase_id → texto castellano legible)
clase_texto = {}
for clase in le.classes_:
    # Intentar extraer texto del SRT correspondiente
    gt_match = ground_truth.get(clase)
    if gt_match:
        clase_texto[clase] = gt_match['transcripcion'][:50]
    else:
        # Limpiar nombre de clase como texto
        texto = clase.replace('_', ' ').replace('-', ' ').title()
        clase_texto[clase] = texto

with open('/content/clase_texto.json', 'w', encoding='utf-8') as f:
    json.dump(clase_texto, f, ensure_ascii=False, indent=2)
print(f'✅ clase_texto.json guardado ({len(clase_texto)} entradas)')
print('Muestra:', list(clase_texto.items())[:5])

In [ ]:
# ── Escribir api/main.py completo ────────────────────────────────────────────
api_code = '''
"""API FastAPI — Traductor LSP en tiempo real (Colab)"""
import io, json, time, base64, asyncio, numpy as np, cv2, torch, sys
from pathlib import Path
from typing import Optional, List
from fastapi import FastAPI, File, UploadFile, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from contextlib import asynccontextmanager
import onnxruntime as ort

ONNX_PATH       = "/content/checkpoints/" + "{nombre_export}" + ".onnx"
LABEL2IDX_PATH  = "/content/label2idx.json"
CLASE_TEXTO_PATH = "/content/clase_texto.json"
N_FRAMES = 30

sess     = None
idx2lbl  = {{}}
cls_text = {{}}

@asynccontextmanager
async def lifespan(app):
    global sess, idx2lbl, cls_text
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    sess = ort.InferenceSession(ONNX_PATH, providers=providers)
    with open(LABEL2IDX_PATH) as f:
        l2i = json.load(f)
    idx2lbl = {{int(v): k for k, v in l2i.items()}}
    if Path(CLASE_TEXTO_PATH).exists():
        with open(CLASE_TEXTO_PATH) as f:
            cls_text = json.load(f)
    print(f"Modelo ONNX listo — {{len(idx2lbl)}} LSP - Vocabulario-palabras")
    yield

app = FastAPI(title="Traductor LSP", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {{"status": "ok", "model_ready": sess is not None, "n_classes": len(idx2lbl)}}

@app.get("/classes")
def classes():
    return {{"classes": sorted(idx2lbl.values()), "total": len(idx2lbl)}}

@app.websocket("/predict/stream")
async def ws_predict(websocket: WebSocket):
    await websocket.accept()
    buf = []
    try:
        while True:
            data    = await websocket.receive_json()
            t0      = time.perf_counter()
            img_b64 = data.get("frame", "")
            frame   = cv2.imdecode(np.frombuffer(base64.b64decode(img_b64), np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                await websocket.send_json({{"error": "frame inválido"}})
                continue
            # Aquí se extraerían landmarks con MediaPipe; por ahora placeholder
            await websocket.send_json({{"status": "ok", "latency_ms": (time.perf_counter()-t0)*1000}})
    except WebSocketDisconnect:
        pass

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''.format(nombre_export=nombre_export)

os.makedirs('/content/api', exist_ok=True)
with open('/content/api/main.py', 'w') as f:
    f.write(api_code)
print('✅ /content/api/main.py generado')

In [ ]:
# Iniciar API en background (Colab)
import subprocess, threading

def run_api():
    subprocess.run(['python', '/content/api/main.py'], capture_output=False)

hilo_api = threading.Thread(target=run_api, daemon=True)
hilo_api.start()
time.sleep(3)

# Verificar health
import urllib.request
try:
    resp = urllib.request.urlopen('http://localhost:8000/health', timeout=5)
    print('✅ API en línea:', resp.read().decode())
except Exception as e:
    print(f'⚠️  API no responde: {e}')

## SECCIÓN 8 — Interfaz Gradio (Demo Completa)

In [ ]:
import gradio as gr
import mediapipe as mp

mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

# Cargar sesión ONNX
providers_gr = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if DEVICE == 'cuda' else ['CPUExecutionProvider']
sess_gr = ort.InferenceSession(onnx_path, providers=providers_gr)
input_name = sess_gr.get_inputs()[0].name

historial_global = []

def extraer_landmarks_mp(frame_rgb, holistic):
    """Extrae keypoints con MediaPipe Holistic → [T=1, KP, 3]."""
    res = holistic.process(frame_rgb)
    coords = []
    listas = [res.left_hand_landmarks, res.right_hand_landmarks, res.pose_landmarks]
    ns     = [21, 21, 33]
    for lm_list, n in zip(listas, ns):
        if lm_list:
            for lm in lm_list.landmark:
                coords.append([lm.x, lm.y, lm.z])
        else:
            coords.extend([[0., 0., 0.]] * n)
    # Ajustar a N_KP esperado por el modelo
    kp = np.array(coords, dtype=np.float32)   # [75, 3] o más
    if kp.shape[0] > N_KP:
        kp = kp[:N_KP]
    elif kp.shape[0] < N_KP:
        pad = np.zeros((N_KP - kp.shape[0], 3), dtype=np.float32)
        kp  = np.concatenate([kp, pad], axis=0)
    return kp

def dibujar_landmarks(frame_bgr, results):
    """Superpone skeleton de manos, pose y cara sobre el frame."""
    frame_out = frame_bgr.copy()
    mp_drawing.draw_landmarks(frame_out, results.left_hand_landmarks,
                               mp_holistic.HAND_CONNECTIONS,
                               mp_styles.get_default_hand_landmarks_style())
    mp_drawing.draw_landmarks(frame_out, results.right_hand_landmarks,
                               mp_holistic.HAND_CONNECTIONS,
                               mp_styles.get_default_hand_landmarks_style())
    mp_drawing.draw_landmarks(frame_out, results.pose_landmarks,
                               mp_holistic.POSE_CONNECTIONS)
    return frame_out

buffer_frames = []

def predecir_frame(frame_rgb):
    """Callback de Gradio: recibe frame, acumula 30, predice."""
    global buffer_frames, historial_global

    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        results = holistic.process(frame_rgb)
        frame_overlay = dibujar_landmarks(frame_bgr, results)
        kp = extraer_landmarks_mp(frame_rgb, holistic)

    kp_norm = normalizar_kp(kp)
    buffer_frames.append(kp_norm)
    if len(buffer_frames) > N_FRAMES:
        buffer_frames.pop(0)

    texto_pred = ''
    confianza  = 0.0
    top3_str   = ''

    if len(buffer_frames) == N_FRAMES:
        kp_seq = np.stack(buffer_frames)           # [T, KP, 3]
        kp_in  = kp_seq[np.newaxis].astype(np.float32)  # [1, T, KP, 3]
        t0     = time.perf_counter()
        logits = sess_gr.run(None, {input_name: kp_in})[0][0]  # [N_CLASES]
        lat    = (time.perf_counter() - t0) * 1000
        probs  = np.exp(logits) / np.exp(logits).sum()
        top_i  = np.argsort(probs)[::-1]
        pred_clase = idx2label.get(int(top_i[0]), '?')
        texto_pred = clase_texto.get(pred_clase, pred_clase.replace('_', ' ').title())
        confianza  = float(probs[top_i[0]])
        top3_str   = '\n'.join(
            f"{i+1}. {clase_texto.get(idx2label.get(int(top_i[i]),'?'), '?')} ({probs[top_i[i]]:.2%})"
            for i in range(min(3, len(top_i)))
        )
        if confianza > 0.4 and (not historial_global or historial_global[-1] != texto_pred):
            historial_global.append(texto_pred)
            historial_global = historial_global[-10:]
        print(f'  → {texto_pred} ({confianza:.2%}) | lat={lat:.0f}ms')

    frame_out_rgb = cv2.cvtColor(frame_overlay, cv2.COLOR_BGR2RGB)
    historial_str = ' | '.join(historial_global[-5:])
    return (
        frame_out_rgb,
        texto_pred if texto_pred else '… esperando señas …',
        f'{confianza:.1%}',
        top3_str,
        historial_str,
    )

print('✅ Funciones de demo definidas')

In [ ]:
# ── Construir interfaz Gradio ────────────────────────────────────────────────
CSS = """
#titulo { text-align: center; font-size: 22px; font-weight: bold; margin-bottom: 10px; }
#traduccion { font-size: 24px; font-weight: bold; color: #1a73e8; min-height: 60px; }
#confianza  { font-size: 16px; color: #555; }
#historial  { font-size: 14px; color: #333; border-top: 1px solid #ddd; padding-top: 8px; }
"""

with gr.Blocks(css=CSS, title='LSP → Castellano') as demo:
    gr.HTML('<div id="titulo">🤟 Traductor LSP → Castellano | En Tiempo Real</div>')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 📷 Cámara — Señas LSP')
            camara = gr.Image(sources=['webcam'], streaming=True,
                               label='Cámara en vivo (con landmarks)',
                               elem_id='cam_overlay')
            video_subido = gr.Video(label='O sube un video MP4', sources=['upload'])

        with gr.Column(scale=1):
            gr.Markdown('### 📝 Traducción en Castellano')
            texto_out     = gr.Textbox(label='Traducción', elem_id='traduccion',
                                        lines=3, interactive=False)
            confianza_out = gr.Textbox(label='Confianza', elem_id='confianza',
                                        interactive=False)
            top3_out      = gr.Textbox(label='Top 3 predicciones', lines=3,
                                        interactive=False)
            gr.Markdown('#### 📜 Historial')
            historial_out = gr.Textbox(label='Últimas señas', elem_id='historial',
                                        interactive=False)

            with gr.Row():
                btn_limpiar = gr.Button('🗑 Limpiar historial')
                btn_export  = gr.Button('💾 Exportar .txt')

    # Evento: cámara en streaming
    camara.stream(
        fn=predecir_frame,
        inputs=[camara],
        outputs=[camara, texto_out, confianza_out, top3_out, historial_out],
    )

    def limpiar():
        global historial_global, buffer_frames
        historial_global = []
        buffer_frames    = []
        return ''

    def exportar_txt():
        ruta = '/content/historial_lsp.txt'
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write('\n'.join(historial_global))
        return gr.File(value=ruta, visible=True)

    btn_limpiar.click(limpiar, outputs=[historial_out])
    archivo_export = gr.File(visible=False)
    btn_export.click(exportar_txt, outputs=[archivo_export])

    gr.Markdown("""
    ---
    **Métricas del sistema:**  
    Latencia objetivo: <200ms | FPS objetivo: ≥24 | Dataset: LSP (Videos + Keypoints + SRT)
    """)

print('✅ Interfaz Gradio definida')

In [ ]:
# ── Lanzar demo ──────────────────────────────────────────────────────────────
demo.launch(
    share=True,           # Genera enlace público ngrok
    debug=False,
    show_error=True,
    server_port=7860,
)
print('✅ Demo lanzada — abrir el enlace público arriba')

## SECCIÓN 9 — Resumen y Archivos Finales

In [ ]:
# ── Copiar entregables a Drive ────────────────────────────────────────────────
import shutil

OUTPUT_DRIVE = os.path.join(DATASET_ROOT, 'LSP_Entregables/')
os.makedirs(OUTPUT_DRIVE, exist_ok=True)

entregables = [
    '/content/checkpoints/lstm_best.pt',
    '/content/checkpoints/stgcn_best.pt',
    f'/content/checkpoints/{nombre_export}.onnx',
    '/content/label2idx.json',
    '/content/clase_texto.json',
    '/content/baseline_results.csv',
    '/content/tabla_comparativa.csv',
    '/content/curvas_aprendizaje.png',
    '/content/matrices_confusion.png',
    '/content/eda_videos.png',
]

for src in entregables:
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DRIVE, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'  ✅ {os.path.basename(src)} → Drive')
    else:
        print(f'  ⚠️  No encontrado: {src}')

print(f'\n📂 Entregables en: {OUTPUT_DRIVE}')

In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────────
print('=' * 65)
print('RESUMEN FINAL — SISTEMA LSP → CASTELLANO')
print('=' * 65)

print(f'\n📊 DATASET')
print(f'   LSP - Palabras: {len(LSP - Palabras)} | LSP - Vocabulario-palabras: {len(conteo_clases)} | SRTs: {len(ground_truth)}')

print(f'\n🏆 RESULTADOS')
print(tabla_final.to_string(index=False))

mejor_dl = 'LSTM Bidir + Attn' if res_lstm['f1_macro'] >= res_stgcn['f1_macro'] else 'ST-GCN'
mejor_f1 = max(res_lstm['f1_macro'], res_stgcn['f1_macro'])
print(f'\n✅ Mejor modelo DL: {mejor_dl} (F1-macro={mejor_f1:.3f})')
print(f'   Supera baseline ({mejor_f1_base:.3f}): {"✅" if mejor_f1 > mejor_f1_base else "❌"}')
print(f'   Latencia ONNX: {np.mean(tiempos_onnx):.1f}ms (objetivo <200ms): {"✅" if np.mean(tiempos_onnx) < 200 else "⚠️"}')

print('\n📦 ARCHIVOS GENERADOS')
for src in entregables:
    existe = '✅' if os.path.exists(src) else '❌'
    print(f'   {existe} {os.path.basename(src)}')

print('\n🎯 SISTEMA LISTO PARA DEMO')
print('   • API FastAPI: http://localhost:8000')
print('   • Interfaz Gradio: ver enlace público arriba')
print('=' * 65)